# Bundle: `ministral-3:3b`, 2 runs back to back

1. pipeline 2 with Arctic-Text2SQL-R1-7B
2. pipeline 1 (judge writes SQL)

All on the 1,000 ClaimDB test claims; results go to `MyDrive/sqljudge/results/`. Runs already finished on Drive are
skipped and half-finished ones resume, so this can pick up from the single-run notebooks.

**Before running:** Runtime → Change runtime type → a fast GPU (G4, A100 or H100) → Save, then Connect.
Check the bottom bar names that GPU, then Run all and click Allow for Drive.

## 1. Settings

In [ ]:
import os
os.environ["JUDGE"] = "ministral-3:3b"
os.environ["NAME"] = "ministral-3b"
os.environ["PLAN"] = "C|a-kore/Arctic-Text2SQL-R1-7B|Arctic-Text2SQL-R1-7B A|-|-"   # runs in order: pipeline|builder tag|builder folder
os.environ["BUILDERS"] = "a-kore/Arctic-Text2SQL-R1-7B"
os.environ["CLAIMS"] = "subsets/test-public.jsonl"
os.environ["DRIVE"] = "/content/drive/MyDrive/sqljudge"
os.environ["RESULTS"] = "/content/drive/MyDrive/sqljudge/results"
os.environ["BIRD_DB_ROOT"] = "/content/bird/dev_databases"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"
os.environ["REPO"] = "/content/claimdb-specialist-sql"
print({k: os.environ[k] for k in ["JUDGE", "PLAN", "RESULTS"]})

## 2. Mount Google Drive (click Allow)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(os.environ["RESULTS"], exist_ok=True)

## 3. Setup: GPU check, Ollama, code, BIRD databases (from the Drive copy), models

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
nohup ollama serve > /content/ollama.log 2>&1 &
until curl -s localhost:11434/api/version > /dev/null; do sleep 1; done; ollama --version
if [ -d "$REPO" ]; then git -C "$REPO" pull -q; else git clone -q https://github.com/Kushagra-Agnihotri/claimdb-specialist-sql.git "$REPO"; fi
pip install -q -r "$REPO/Navtej/requirements.txt"
git -C "$REPO" log --oneline -1
mkdir -p "$DRIVE/bird" /content/bird
[ -f "$BIRD_DB_ROOT/formula_1/formula_1.sqlite" ] || unzip -q -o "$DRIVE/bird/dev_databases.zip" -d /content/bird
rm -rf /content/bird/__MACOSX; echo "BIRD databases: $(ls "$BIRD_DB_ROOT" | wc -l)"
echo "pulling $JUDGE"; ollama pull "$JUDGE" > /dev/null 2>&1
for b in $BUILDERS; do echo "pulling $b"; ollama pull "$b" > /dev/null 2>&1; done
ollama list

## 4. The runs, one after the other

In [ ]:
%%bash
cd "$REPO/Navtej"
for step in $PLAN; do
  IFS='|' read -r pipe btag bname <<< "$step"
  echo "=== $(date '+%H:%M') starting $pipe $bname"
  if [ "$pipe" = A ]; then
    ./pipeline1_A/run_A.sh --model "$JUDGE" --name "$NAME" --claims "$CLAIMS" --db-root "$BIRD_DB_ROOT" --results-root "$RESULTS"
  else
    ./pipeline2_C/run_C.sh --model "$JUDGE" --name "$NAME" --builder "$btag" --builder-name "$bname" \
      --claims "$CLAIMS" --db-root "$BIRD_DB_ROOT" --results-root "$RESULTS"
  fi
done

## 5. Score all runs of this judge

In [ ]:
%%bash
cd "$REPO/Navtej"
for step in $PLAN; do
  IFS='|' read -r pipe btag bname <<< "$step"
  if [ "$pipe" = A ]; then ./pipeline1_A/score_A.sh --name "$NAME" --claims "$CLAIMS" --results-root "$RESULTS"
  else ./pipeline2_C/score_C.sh --name "$NAME" --builder-name "$bname" --claims "$CLAIMS" --results-root "$RESULTS"; fi
done

## Progress (optional; or just look at the files on Drive)

In [ ]:
%%bash
for f in $(find "$RESULTS" -path "*/$NAME/*" -name results.jsonl); do echo "$(wc -l < "$f") / 1000  $f"; done